# Customer Churn Prediction – Baseline ML Notebook

**RabTech Academy | AI/ML Internship | Task 02**

## Objective
Build a simple, responsible baseline model to explore whether customer activity and subscription information can help identify churn.

**Dataset:** `customer-churn-training.csv`

**Important limitation:** This dataset contains only 12 records. Results are for learning and workflow demonstration only; they must not be treated as reliable real-world performance or used to make decisions about actual customers.


## 1. Import libraries

We use Pandas for data handling, Matplotlib and Seaborn for visualization, and Scikit-learn for preprocessing, modelling and evaluation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

RANDOM_STATE = 42


## 2. Load the dataset

Upload `customer-churn-training.csv` to the notebook environment (or keep it in the same folder as this notebook).


In [ ]:
# In Google Colab, uncomment and run these two lines to upload the CSV:
# from google.colab import files
# files.upload()

DATA_PATH = "customer-churn-training.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())


## 3. Understand the dataset

The target column is `churned`:
- `1` = customer churned
- `0` = customer did not churn

`customer_id` is an identifier, not a meaningful predictive feature, so it will be excluded from model training.


In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\\nData types:")
print(df.dtypes)

print("\\nMissing values:")
print(df.isnull().sum())

print("\\nDuplicate rows:", df.duplicated().sum())

print("\\nTarget distribution:")
print(df["churned"].value_counts().sort_index())

print("\\nTarget proportions:")
print(df["churned"].value_counts(normalize=True).sort_index())


In [ ]:
# Visualize the target distribution
target_counts = df["churned"].value_counts().sort_index()

plt.figure(figsize=(6, 4))
sns.barplot(x=target_counts.index.astype(str), y=target_counts.values)
plt.title("Customer Churn Distribution")
plt.xlabel("Churned (0 = No, 1 = Yes)")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.show()


## 4. Define features and target

We exclude `customer_id` to avoid learning from an arbitrary identifier. We also keep the target separate from the input features.


In [ ]:
target_column = "churned"
excluded_columns = ["customer_id", target_column]

X = df.drop(columns=excluded_columns)
y = df[target_column]

numeric_features = [
    "tenure_months",
    "support_tickets",
    "monthly_spend_inr",
    "last_login_days"
]
categorical_features = ["plan_type"]

print("Input features:", X.columns.tolist())
print("Target:", target_column)


## 5. Preprocess the features

Numeric features are standardized. The categorical `plan_type` feature is one-hot encoded. The preprocessing is placed inside a Scikit-learn Pipeline so it is fitted separately within each cross-validation fold.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

print("Preprocessing and model pipeline created.")


## 6. Establish a simple majority-class baseline

A majority-class classifier always predicts the most frequent class in its training data. This gives us a simple reference point against which to compare Logistic Regression.


In [ ]:
loo = LeaveOneOut()

baseline_model = DummyClassifier(strategy="most_frequent")
baseline_predictions = cross_val_predict(baseline_model, X, y, cv=loo)

print("Majority baseline accuracy:",
      round(accuracy_score(y, baseline_predictions), 3))
print("Baseline predictions:", baseline_predictions)


## 7. Train and evaluate Logistic Regression with Leave-One-Out Cross-Validation

Leave-One-Out Cross-Validation trains on 11 records and tests on the remaining record, repeating this process 12 times. Each record is therefore tested once.

This makes efficient use of a very small dataset, but it does **not** solve the problem of having too little data. The estimates will be highly uncertain.


In [ ]:
model_predictions = cross_val_predict(model, X, y, cv=loo)

print("Logistic Regression – Leave-One-Out results")
print("Accuracy :", round(accuracy_score(y, model_predictions), 3))
print("Precision:", round(precision_score(y, model_predictions, zero_division=0), 3))
print("Recall   :", round(recall_score(y, model_predictions, zero_division=0), 3))
print("F1-score :", round(f1_score(y, model_predictions, zero_division=0), 3))

print("\\nClassification report:")
print(classification_report(
    y, model_predictions,
    labels=[0, 1],
    target_names=["Not churned", "Churned"],
    zero_division=0
))


## 8. Confusion matrix

The confusion matrix helps us inspect correct and incorrect predictions. In churn prediction, a false negative means a customer who churned was predicted as not churned. A false positive means a customer who did not churn was predicted as churned.


In [ ]:
cm = confusion_matrix(y, model_predictions, labels=[0, 1])

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Not churned", "Churned"],
    yticklabels=["Not churned", "Churned"]
)
plt.xlabel("Predicted label")
plt.ylabel("Actual label")
plt.title("Logistic Regression – Confusion Matrix")
plt.tight_layout()
plt.show()


## 9. Compare with the majority baseline

Compare the two approaches using the same cross-validation predictions. Do not select a model based only on accuracy; inspect recall, precision and the types of errors as well.


In [ ]:
comparison = pd.DataFrame({
    "Model": ["Majority baseline", "Logistic Regression"],
    "Accuracy": [
        accuracy_score(y, baseline_predictions),
        accuracy_score(y, model_predictions)
    ],
    "Precision": [
        precision_score(y, baseline_predictions, zero_division=0),
        precision_score(y, model_predictions, zero_division=0)
    ],
    "Recall": [
        recall_score(y, baseline_predictions, zero_division=0),
        recall_score(y, model_predictions, zero_division=0)
    ],
    "F1-score": [
        f1_score(y, baseline_predictions, zero_division=0),
        f1_score(y, model_predictions, zero_division=0)
    ]
})

display(comparison.round(3))


## 10. Responsible interpretation and limitations

1. **Very small sample:** There are only 12 customer records. Cross-validation results can change substantially with additional records.
2. **Representation:** The dataset may not represent the full customer population or all customer groups.
3. **Privacy:** `customer_id` is excluded from modelling. Do not publish identifiable customer data.
4. **Potential leakage:** Confirm that every feature was collected before the churn prediction decision. The supplied file does not document collection dates.
5. **Fairness:** `plan_type` may be associated with different customer experiences. This sample is too small for meaningful fairness conclusions.
6. **Human oversight:** Predictions must not automatically cancel accounts, change prices or deny services.
7. **Deployment:** Do not deploy this model for real customer decisions without a larger, representative dataset, documented permission, temporal validation, monitoring and appropriate review.

Probability calibration is not assessed because this dataset is far too small to support a meaningful calibration analysis.


## 11. Conclusion

This notebook demonstrates a basic customer churn prediction workflow using data inspection, preprocessing, a majority-class baseline, Logistic Regression and Leave-One-Out Cross-Validation.

The results are preliminary because the supplied dataset contains only 12 records. The notebook is a learning demonstration and does not establish that the model is accurate, fair or suitable for real-world deployment.

**Next step:** Add the notebook, the supplied CSV (only if its sharing permission allows it), and the supporting memo, data card and risk register to the project repository.
